# Topographic-control transition case studies

This notebook automatically identifies long-lived eddies that spend sustained periods in both planetary- and topographic-PV-gradient regimes. These tracks provide an internal reference: changes in tilt can be compared within the same eddy as it approaches, crosses, or leaves strong bathymetric gradients.

The diagnostic is $R = \log(|\nabla q|_{topographic}/|\nabla q|_{planetary})$, where $R<0$ is planetary-dominant and $R>0$ is topographic-dominant. Ranking is performed separately for anticyclonic (AE) and cyclonic (CE) eddies so that the initial comparison contains equal numbers of each polarity.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next(
    (p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()),
    None,
)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or one of its subfolders.')
CASE_ROOT = ANALYSIS_ROOT / 'case_studies'
for path in (CASE_ROOT, ANALYSIS_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import seacofs_tilt_tools as tilt
from case_study_tools import (
    TransitionConfig,
    plot_topographic_transition,
    rank_topographic_transitions,
    select_top_cases,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Load the vertically checked eddy dataset

Core-mean bathymetry and gradients are used because the process should represent the bathymetry experienced across the eddy core, rather than a single centre grid cell. This is the expensive step. Set `CORE_MEAN_FOR_SCREENING = False` for a quick preliminary screen, then rerun selected cases with core means before interpreting them.

In [ ]:
CORE_MEAN_FOR_SCREENING = True

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, _ = tilt.load_tilt_tables(paths)
df_eddies = tilt.add_region_labels(df_eddies, grid)
df_eddies = tilt.add_pv_gradient_terms(
    df_eddies, grid, core_mean=CORE_MEAN_FOR_SCREENING
)

required = {'Eddy', 'Day', 'Cyc', 'TiltDis', 'TiltDir', 'topo_plan_ratio'}
missing = required - set(df_eddies.columns)
if missing:
    raise KeyError(f'Missing required columns: {sorted(missing)}')
if df_eddies.duplicated(['Eddy', 'Day']).any():
    raise ValueError('Expected one row per Eddy-Day.')

print(f"{len(df_eddies):,} observations from {df_eddies.Eddy.nunique():,} eddies")
display(df_eddies.groupby('Cyc').agg(observations=('Eddy', 'size'), eddies=('Eddy', 'nunique')))

## 2. Define an ideal transition candidate

The defaults require a lifetime of at least 100 days, at least 10 valid observations in each regime, a run of at least five consecutive smoothed observations in each regime, and at least 20 valid tilt estimates. A centred seven-observation rolling median suppresses isolated crossings.

Eligible eddies are scored using lifetime (20%), balance between regimes (25%), shorter of the two sustained runs (20%), separation of regime medians (15%), few clean crossings (10%), and tilt-data coverage (10%). All score components are percentile-ranked within AE or CE.

In [ ]:
config = TransitionConfig(
    smooth_window=7,
    min_periods=5,
    min_lifetime_days=100,
    min_regime_observations=10,
    min_sustained_run=5,
    min_tilt_observations=20,
    regime_margin=0.0,
)

df_transition, ranking = rank_topographic_transitions(df_eddies, config)
eligibility = ranking.groupby('Cyc')['eligible'].agg(candidates='size', eligible='sum')
display(eligibility)

## 3. Inspect the ranked candidates

High scores indicate useful presentation cases, not proof that topography caused their tilt. Before selecting a final case, inspect the trajectory and confirm that the transition corresponds to a coherent encounter with shelf or slope bathymetry rather than a boundary artefact or a brief return crossing.

In [ ]:
ranking_columns = [
    'Eddy', 'Cyc', 'Region', 'transition_score', 'lifetime_days',
    'planetary_fraction', 'topographic_fraction', 'regime_balance',
    'longest_planetary_run', 'longest_topographic_run',
    'regime_separation', 'sign_changes', 'tilt_coverage', 'median_tilt_km',
]
for cyc in ['AE', 'CE']:
    print(f'\nTop {cyc} transition candidates')
    display(ranking.loc[(ranking.Cyc == cyc) & ranking.eligible, ranking_columns].head(20).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
eligible = ranking[ranking.eligible].copy()
for ax, cyc in zip(axes, ['AE', 'CE']):
    part = eligible[eligible.Cyc == cyc]
    sc = ax.scatter(part.regime_balance, part.sustained_run_min, c=part.transition_score,
                    s=np.clip(part.lifetime_days, 50, 400), cmap='viridis',
                    vmin=0, vmax=1, alpha=0.75, edgecolor='white')
    for row in part.nlargest(5, 'transition_score').itertuples():
        ax.annotate(str(row.Eddy), (row.regime_balance, row.sustained_run_min), fontsize=8)
    ax.set(title=cyc, xlabel='Regime balance', ylabel='Shorter sustained run (observations)')
fig.colorbar(sc, ax=axes, label='Transition score')
fig.suptitle('Eligible planetary–topographic transition candidates');

## 4. Select equal numbers of AE and CE cases

The default selection uses the three highest-ranked eligible eddies of each polarity. Edit `selected` after examining the candidate figures if a track is affected by coastline geometry, missing data, or an unhelpful repeated crossing.

In [ ]:
N_PER_POLARITY = 3
selected = select_top_cases(ranking, n_per_polarity=N_PER_POLARITY)
selected

## 5. Plot the selected cases

Vertical dotted lines mark crossings of the smoothed dominance ratio. The map is coloured by the same ratio: blue is planetary dominance and red is topographic dominance. Direction panels use compass bearings, with 0/360° north, 90° east, 180° south and 270° west.

In [ ]:
for cyc in ['AE', 'CE']:
    for eddy_id in selected[cyc]:
        track = df_transition[df_transition.Eddy == eddy_id].copy()
        fig, _, _ = plot_topographic_transition(
            track, grid, title=f'{cyc} eddy {eddy_id}: planetary–topographic transition'
)
        plt.show()

## 6. Within-eddy regime comparison

This descriptive table compares the same eddy between its planetary- and topographic-dominant periods. It is intended to guide case interpretation. Differences can coincide with changing age, location, strength, stratification, or background flow and should not be presented as an isolated causal estimate.

In [ ]:
selected_ids = selected['AE'] + selected['CE']
comparison = df_transition[df_transition.Eddy.isin(selected_ids)].copy()
comparison['regime'] = np.select(
    [comparison.planetary_dominant, comparison.topographic_dominant],
    ['planetary', 'topographic'], default='transition'
)
comparison['slope'] = np.hypot(comparison.dhdx, comparison.dhdy)
regime_summary = (
    comparison[comparison.regime != 'transition']
    .groupby(['Cyc', 'Eddy', 'regime'])
    .agg(observations=('Day', 'size'), tilt_median_km=('TiltDis', 'median'),
         tilt_iqr_km=('TiltDis', lambda x: x.quantile(.75) - x.quantile(.25)),
         depth_median_m=('h', 'median'), slope_median=('slope', 'median'),
         ratio_median=('topo_plan_ratio_smooth', 'median'), rossby_median=('Ro', 'median'))
    .round(3)
)
display(regime_summary)

## Interpretation checklist

A convincing topographic case should show: (1) a sustained rather than single-day change in dominance; (2) a spatially coherent encounter with shelf/slope bathymetry; (3) a corresponding change in tilt magnitude and/or bearing; (4) a total PV-gradient response consistent with the changing topographic contribution; and (5) no obvious loss of tilt-data coverage at the transition.

The CE–AE comparison is initially balanced. If the ranked and visually checked CEs consistently show clearer responses, the final presentation can then emphasize CEs while retaining one or more AE counterexamples.